# Change from V2.0:
## Updated Secondary Label detection and allication in labeling stage

In [2]:
# %% [markdown]
# # RQ2 — Step 0: Clone repositories
# Reads a CSV with column `repo_url`, clones/fetches into CLONE_ROOT,
# and writes a manifest with basic metadata.

# %%
from __future__ import annotations
import csv, subprocess, sys, json, time
from pathlib import Path
from typing import Optional, List
from pathlib import Path

# -----------------------------
# Config (edit as needed)
# -----------------------------


# Use a raw string r"..." for Windows paths with spaces
WORK_ROOT    = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2")
URL_LIST_CSV = WORK_ROOT / "URL_List.csv"   # put your CSV here
CLONE_ROOT   = WORK_ROOT / "clones"         # repos will clone here
MANIFEST_CSV = WORK_ROOT / "clones_manifest.csv"

WORK_ROOT.mkdir(parents=True, exist_ok=True)
CLONE_ROOT.mkdir(parents=True, exist_ok=True)


# Create dirs
WORK_ROOT.mkdir(parents=True, exist_ok=True)
CLONE_ROOT.mkdir(parents=True, exist_ok=True)

# %%
def sh(cmd: List[str], cwd: Optional[Path] = None, check: bool = True) -> subprocess.CompletedProcess:
    return subprocess.run(cmd, cwd=cwd, check=check, capture_output=True, text=True)

def repo_dir_name_from_url(url: str) -> str:
    # e.g. https://github.com/owner/name(.git) -> owner__name
    base = url.split("//")[-1]
    parts = base.split("/")
    if len(parts) >= 3:
        owner = parts[-2]
        name  = parts[-1].replace(".git", "")
        return f"{owner}__{name}"
    return base.replace("/", "__").replace(".git", "")

def ensure_cloned(url: str, dest_root: Path) -> Path:
    dest_root.mkdir(parents=True, exist_ok=True)
    d = dest_root / repo_dir_name_from_url(url)
    if d.exists() and (d / ".git").exists():
        # Refresh remote info (best-effort)
        try:
            sh(["git", "fetch", "--all", "--tags", "--prune"], cwd=d)
        except Exception:
            pass
        return d
    sh(["git", "clone", "--no-tags", "--filter=blob:none", "--recurse-submodules=no", url, str(d)])
    return d

def get_total_commits(repo_dir: Path) -> int:
    cp = sh(["git", "rev-list", "--all", "--count"], cwd=repo_dir)
    return int(cp.stdout.strip() or "0")

# %%
assert URL_LIST_CSV.exists(), f"CSV not found: {URL_LIST_CSV}"

rows, ok, fail = [], 0, 0
with URL_LIST_CSV.open(newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        url = (row.get("repo_url") or "").strip()
        if not url:
            continue
        t0 = time.time()
        rec = {"repo_url": url, "dir": None, "status": "unknown", "seconds": None, "total_commits": None, "error": ""}
        try:
            d = ensure_cloned(url, CLONE_ROOT)
            rec["dir"] = str(d)
            rec["total_commits"] = get_total_commits(d)
            rec["status"] = "ok"
            ok += 1
        except subprocess.CalledProcessError as e:
            rec["status"] = "error"
            rec["error"]  = (e.stderr or e.stdout or str(e)).strip()[:2000]
            fail += 1
        rec["seconds"] = round(time.time() - t0, 2)
        rows.append(rec)
        print(f"[{rec['status']}] {url} -> {rec['dir']} ({rec['seconds']}s)")

# %%
# Write manifest
MANIFEST_CSV.parent.mkdir(parents=True, exist_ok=True)
with MANIFEST_CSV.open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()) if rows else ["repo_url","dir","status","seconds","total_commits","error"])
    w.writeheader()
    w.writerows(rows)

print(f"\nDone. OK={ok}, FAIL={fail}. Manifest: {MANIFEST_CSV}")


[ok] https://github.com/connectbot/connectbot -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\connectbot__connectbot (1.64s)
[ok] https://github.com/robolectric/robolectric -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\robolectric__robolectric (7.22s)
[ok] https://github.com/opendocument-app/OpenDocument.droid -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\opendocument-app__OpenDocument.droid (4.29s)
[ok] https://github.com/maxpower47/PinDroid -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\maxpower47__PinDroid (2.95s)
[ok] https://github.com/Rajawali/Rajawali -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\Rajawali__Rajawali (6.46s)
[ok] https://github.com/cgeo/cgeo -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\cgeo__cgeo (24.03s)
[ok] https://github.com/OneBusAway/onebusaway-android -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\OneBusAway__onebus

In [4]:
# %% [markdown]
# RQ2 — Step 1: Mine commit snapshots (inclusive CI patterns + cutoff)
# Scans cloned repos for commits touching CI/YAML/Gradle/scripts and writes per-repo JSONL snapshots.
# - Windows-safe UTF-8 decoding for all git calls (prevents UnicodeDecodeError).
# - Robust fallbacks so a wonky repo doesn't stop the whole run.
# - Inclusive CI vendor path detection (Travis/AppVeyor/CircleCI/GHA/GitLab/Jenkins/etc.).
# - Reduced script false positives (extensions only).
# - Canonical headless tag (no duplicate "no_window").
# - Invocation tags also detected in Gradle DSL.
# - Richer AGP & Orchestrator detection.
# - Commit date cutoff: include commits up to end of Aug 10, 2025 (America/Toronto).

from __future__ import annotations

import os
import re
import json
import subprocess
import datetime as dt
from pathlib import Path
from typing import List, Tuple, Optional, Dict, Any, Set

# ---- Optional tz support (Py 3.9+) ----
try:
    from zoneinfo import ZoneInfo  # Python 3.9+
except Exception:
    ZoneInfo = None

# -----------------------------
# Config (edit as needed)
# -----------------------------
WORK_ROOT      = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2")
CLONE_ROOT     = WORK_ROOT / "clones"
SNAPSHOT_DIR   = WORK_ROOT / "snapshots"   # per-repo JSONL output here
MAX_COMMITS_PER_REPO = 0                   # 0 = no limit

SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Commit cutoff (America/Toronto) ----
CUTOFF_TZ_NAME = "America/Toronto"
_CUTOFF_DATE   = (2025, 8, 10, 23, 59, 59)  # YYYY, M, D, H, M, S local to Toronto
if ZoneInfo is not None:
    _tz = ZoneInfo(CUTOFF_TZ_NAME)
    _local_dt = dt.datetime(*_CUTOFF_DATE, tzinfo=_tz)
    CUTOFF_EPOCH = int(_local_dt.timestamp())
    # Format for `git --before`, e.g., "2025-08-10 23:59:59 -0400"
    CUTOFF_BEFORE_STR = _local_dt.strftime("%Y-%m-%d %H:%M:%S %z")
else:
    # Fallback: On Aug 10, 2025 Toronto is EDT (UTC-4)
    _utc_dt = dt.datetime(2025, 8, 11, 3, 59, 59, tzinfo=dt.timezone.utc)
    CUTOFF_EPOCH = int(_utc_dt.timestamp())
    CUTOFF_BEFORE_STR = "2025-08-10 23:59:59 -0400"

print(f"[cutoff] Using commit cutoff <= {CUTOFF_BEFORE_STR} (epoch={CUTOFF_EPOCH})")

# Optional YAML support (safe to skip if you don't need deep YAML parsing)
try:
    import yaml  # pip install pyyaml
except Exception:
    yaml = None

# -----------------------------
# Subprocess helper (Windows-safe UTF-8)
# -----------------------------
def sh(cmd: List[str], cwd: Optional[Path] = None, check: bool = True) -> subprocess.CompletedProcess:
    """
    Run a command and return CompletedProcess with UTF-8 decoding and error replacement.
    Prevents UnicodeDecodeError on Windows when reading git output.
    """
    env = os.environ.copy()
    # Disable the pager portably
    env["GIT_PAGER"] = ""
    return subprocess.run(
        cmd,
        cwd=str(cwd) if cwd is not None else None,
        check=check,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        encoding="utf-8",   # force UTF-8 decode
        errors="replace",   # never crash on odd bytes
        env=env,
    )

# -----------------------------
# Relevant file surfaces (INCLUSIVE CI VENDOR LIST)
# -----------------------------
# Universal CI vendor patterns (case-insensitive; supports yml|yaml; handles Jenkinsfile)
CI_VENDOR_PATTERNS = [
    re.compile(r'(?i)(?:^|/)\.travis\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)\.appveyor\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)appveyor\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)circle\.yml$'),
    re.compile(r'(?i)(?:^|/)\.circleci/config\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)azure-pipelines\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)\.github/workflows/.*\.(yml|yaml)$'),
    re.compile(r'(?i)(?:^|/)bitbucket-pipelines\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)\.gitlab-ci\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)Jenkinsfile(?:\.\w+)?$'),   # Jenkinsfile or Jenkinsfile.yml
    re.compile(r'(?i)(?:^|/)bitrise\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)bamboo\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)codeship-services\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)\.gocd\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)\.cirrus\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)wercker\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)semaphore\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)codemagic\.ya?ml$'),
]
# Recall-oriented fallback for generic CI directories (also include ".jenkins")
GENERIC_CI_DIRS = re.compile(r'(?i)(?:^|/)(?:ci|\.ci|\.jenkins)(?:/|$)')

def is_ci_file(p: str) -> bool:
    if not p:
        return False
    for rx in CI_VENDOR_PATTERNS:
        if rx.search(p):
            return True
    return bool(GENERIC_CI_DIRS.search(p))

# Gradle + script surfaces
GRADLE_FILES = [
    "build.gradle", "build.gradle.kts",
    "settings.gradle", "settings.gradle.kts",
    "gradle.properties",
    "gradle/wrapper/gradle-wrapper.properties",
]
SCRIPT_EXTS = {".sh", ".bash", ".zsh", ".py", ".bat", ".cmd", ".ps1", ".psm1"}

def is_gradle_file(p: str) -> bool:
    lp = p.lower()
    return any(lp.endswith(x) for x in (f.lower() for f in GRADLE_FILES))

def is_script_file(p: str) -> bool:
    lp = p.lower()
    return any(lp.endswith(ext) for ext in SCRIPT_EXTS)

def touched_relevant(paths: List[str]) -> bool:
    for p in paths:
        if not p.strip():
            continue
        if is_ci_file(p) or is_gradle_file(p) or is_script_file(p):
            return True
    return False

# -----------------------------
# Git helpers
# -----------------------------
def list_relevant_commits(repo_dir: Path) -> List[Tuple[str, int, List[str]]]:
    """
    Returns list of (sha, unix_ts, [changed_paths]) for commits that touched relevant files,
    limited to commits with committer date <= CUTOFF_EPOCH. Oldest -> newest order.
    """
    cp = sh([
        "git", "-c", "i18n.logOutputEncoding=UTF-8", "-c", "core.quotepath=off",
        "log", "--all",
        "--before", CUTOFF_BEFORE_STR,     # primary bound (local tz string)
        "--name-only", "--pretty=%H%x09%ct"
    ], cwd=repo_dir)
    results: List[Tuple[str, int, List[str]]] = []
    sha: Optional[str] = None
    ts: Optional[int] = None
    changed: List[str] = []
    for line in cp.stdout.splitlines():
        if re.match(r"^[0-9a-f]{40}\t\d+$", line):
            # finalize the previous block
            if sha is not None and ts is not None and ts <= CUTOFF_EPOCH and touched_relevant(changed):
                results.append((sha, ts, changed))
            sha, ts_s = line.split("\t", 1)
            ts = int(ts_s)
            changed = []
        else:
            if line.strip():
                changed.append(line.strip())
    # finalize the last block
    if sha is not None and ts is not None and ts <= CUTOFF_EPOCH and touched_relevant(changed):
        results.append((sha, ts, changed))
    results.reverse()  # oldest -> newest
    return results

def git_show(repo_dir: Path, sha: str, path: str) -> Optional[str]:
    try:
        cp = sh(["git", "show", f"{sha}:{path}"], cwd=repo_dir)
        return cp.stdout
    except subprocess.CalledProcessError:
        return None

def git_subject(repo_dir: Path, sha: str) -> str:
    try:
        cp = sh(["git", "-c", "i18n.logOutputEncoding=UTF-8", "show", "-s", "--format=%s", sha],
                cwd=repo_dir, check=False)
        return (cp.stdout or "").strip()
    except Exception:
        return ""

# -----------------------------
# Heuristic extractors
# -----------------------------
RE_INT = re.compile(r"\d+")

YAML_KEYS = {
    "api": ["api-level","apilevel","api_level"],
    "abi": ["abi","arch","cpu","abi_filters","abi-filter"],
    "system_image": ["system-image","target","systemimage"],
    "device": ["device","avd-name","avd","device-profile","model","hardwareProfile"],
    "orchestrator": ["orchestrator","android-test-orchestrator","use-orchestrator"],
    "wait": ["wait-for-boot","wait_for_boot"],
    "timeouts": ["emulator-boot-timeout","timeout","test-timeout","emulator_timeout"],
    "retries": ["retry","retries","max-retries"],
    "matrix": ["matrix","strategy"],
    "runner_os": ["runs-on","machine","image"],
    "jdk": ["java-version","jdk","java","distribution"],
    "invocation": ["run","gradle_args","gradlew_args","task","tasks"],
    "thirdparty": ["browserstack","saucelabs","firebase","bitbar","kobiton","testlab","devicefarm"],
}

# --- Simplified invocation classification (2D: style + tags) ---

DIY_RX = re.compile(
    r"(?:\bemulator(?:\.bat)?\s-|"
    r"\bavdmanager\b|"
    r"\bsdkmanager\b|"
    r"\bcreate\s+avd\b|"
    r"\badb\s+(?:-s\s+\S+\s+)?wait-for-device\b|"
    r"\badb\s+shell\s+getprop\s+sys\.boot_completed\b|"
    r"\bqemu\b)",
    re.I,
)

GMD_RX = re.compile(
    r"(?:\bmanaged\s+device\b|\bgradle\s+managed\s+device\b|\bPixel\w*Api\d+\b)",
    re.I,
)

GMD_GRADLE_RX = re.compile(
    r"(?:\bmanagedDevices\s*\{|testOptions\s*\{[^}]*devices)",
    re.I | re.S,
)

CONNECTED_RX = re.compile(r"\bconnectedAndroidTest\b", re.I)

# Canonical tag set from YAML CLI snippets
TAG_PATTERNS_YAML = {
    "headless":    re.compile(r"-no-window|-headless", re.I),   # canonical
    "gpu":         re.compile(r"-gpu\s+\w+", re.I),
    "concurrency": re.compile(r"--parallel|\bmax-workers\b|\borg\.gradle\.workers\.max\b", re.I),
    "sharding":    re.compile(r"\bnumShards\b|\bshard(?:ing|Index)?\b", re.I),
    "instr_args":  re.compile(r"-Pandroid\.testInstrumentationRunnerArguments\.", re.I),
}

# Tags detectable in Gradle DSL / properties
TAG_PATTERNS_GRADLE = {
    "sharding":    re.compile(r"testInstrumentationRunnerArguments(?:\[[\"']|\.)(?:numShards|shardIndex)", re.I),
    "instr_args":  re.compile(r"testInstrumentationRunnerArguments", re.I),
    "concurrency": re.compile(r"\borg\.gradle\.workers\.max\b|\bmaxWorkers\b", re.I),
}

def _collect_tags_from_text(text: str, patterns: Dict[str, re.Pattern]) -> Set[str]:
    return {name for name, rx in patterns.items() if rx.search(text or "")}

def classify_invocation_from_texts(yaml_snippets: List[str], gradle_texts: List[str]) -> tuple[str, List[str]]:
    """
    Returns (invocation_style, invocation_tags)
    - yaml_snippets: values from 'run'/'task'/'args' collected from YAML
    - gradle_texts: full text of any Gradle files in the same snapshot (optional)
    """
    hay_yaml = "\n".join(s for s in yaml_snippets if s)
    hay_gradle = "\n".join(gradle_texts or [])

    gmd_hit = bool(GMD_RX.search(hay_yaml)) or any(GMD_GRADLE_RX.search(t or "") for t in gradle_texts or [])
    diy_hit = bool(DIY_RX.search(hay_yaml) or DIY_RX.search(hay_gradle))
    connected_hit = bool(CONNECTED_RX.search(hay_yaml) or CONNECTED_RX.search(hay_gradle))

    if gmd_hit:
        style = "gmd"
    elif diy_hit:
        style = "diy"
    elif connected_hit:
        style = "gradle_connected"
    else:
        style = "unknown"

    tags_yaml = _collect_tags_from_text(hay_yaml, TAG_PATTERNS_YAML)
    tags_gradle = set()
    for t in gradle_texts or []:
        tags_gradle |= _collect_tags_from_text(t, TAG_PATTERNS_GRADLE)

    tags = sorted(tags_yaml | tags_gradle)
    return style, tags

# -----------------------------
# YAML extractor
# -----------------------------
def extract_from_yaml_text(text: str, gradle_texts: Optional[List[str]] = None) -> Dict[str, Any]:
    if yaml is None:
        return {}
    try:
        docs = list(yaml.safe_load_all(text))
    except Exception:
        docs = []
    out = {
        # real features
        "api_levels": set(), "abis": set(), "system_images": set(), "device_profiles": set(),
        "orchestrator": None, "wait_for_boot": None, "timeouts": {}, "retries": None,
        "matrix_axes": set(), "runner_os": None, "jdk": None,
        # study-defined (simplified)
        "invocation_style": "unknown", "invocation_tags": set(),
        "thirdparty_refs": set(),
    }
    # temp bucket: raw run/task/args strings for classification
    _invocation_snippets: List[str] = []

    def scan_obj(obj):
        if isinstance(obj, dict):
            for k, v in obj.items():
                lk = str(k).lower()
                if lk in (x.lower() for x in YAML_KEYS["api"]):
                    if isinstance(v, list):
                        for x in v:
                            if isinstance(x, (int, str)) and RE_INT.search(str(x)):
                                out["api_levels"].add(int(RE_INT.search(str(x)).group()))
                    elif isinstance(v, (int, str)):
                        m = RE_INT.search(str(v))
                        if m:
                            out["api_levels"].add(int(m.group()))
                if lk in (x.lower() for x in YAML_KEYS["abi"]):
                    vals = v if isinstance(v, list) else [v]
                    for x in vals:
                        if isinstance(x, str):
                            out["abis"].add(x.strip())
                if lk in (x.lower() for x in YAML_KEYS["system_image"]):
                    vals = v if isinstance(v, list) else [v]
                    for x in vals:
                        if isinstance(x, str):
                            out["system_images"].add(x.strip())
                if lk in (x.lower() for x in YAML_KEYS["device"]):
                    vals = v if isinstance(v, list) else [v]
                    for x in vals:
                        if isinstance(x, str):
                            out["device_profiles"].add(x.strip())
                if lk in (x.lower() for x in YAML_KEYS["orchestrator"]):
                    if isinstance(v, bool):
                        out["orchestrator"] = v
                    elif isinstance(v, str):
                        out["orchestrator"] = v.lower() in ("1", "true", "yes", "on")
                if lk in (x.lower() for x in YAML_KEYS["wait"]):
                    if isinstance(v, bool):
                        out["wait_for_boot"] = v
                    elif isinstance(v, str):
                        out["wait_for_boot"] = v.lower() in ("1", "true", "yes", "on")
                if lk in (x.lower() for x in YAML_KEYS["timeouts"]):
                    out["timeouts"][k] = v
                if lk in (x.lower() for x in YAML_KEYS["retries"]):
                    try:
                        out["retries"] = int(RE_INT.search(str(v)).group())
                    except Exception:
                        pass
                if lk in (x.lower() for x in YAML_KEYS["matrix"]):
                    if isinstance(v, dict):
                        for ax, _vals in v.items():
                            out["matrix_axes"].add(str(ax))
                if lk in (x.lower() for x in YAML_KEYS["runner_os"]):
                    out["runner_os"] = str(v)
                if lk in (x.lower() for x in YAML_KEYS["jdk"]):
                    out["jdk"] = str(v)
                if lk in (x.lower() for x in YAML_KEYS["invocation"]):
                    _invocation_snippets.append(str(v))
                if lk in (x.lower() for x in YAML_KEYS["thirdparty"]):
                    out["thirdparty_refs"].add(lk)
                if isinstance(v, (dict, list)):
                    scan_obj(v)
        elif isinstance(obj, list):
            for x in obj:
                scan_obj(x)

    for d in docs:
        scan_obj(d)

    # classify invocation using YAML snippets + Gradle context (if provided)
    style, tags = classify_invocation_from_texts(_invocation_snippets, gradle_texts or [])
    out["invocation_style"] = style
    out["invocation_tags"].update(tags)

    # convert sets to sorted lists for JSON
    out["api_levels"]      = sorted(out["api_levels"])
    out["abis"]            = sorted(out["abis"])
    out["system_images"]   = sorted(out["system_images"])
    out["device_profiles"] = sorted(out["device_profiles"])
    out["matrix_axes"]     = sorted(out["matrix_axes"])
    out["invocation_tags"] = sorted(out["invocation_tags"])
    out["thirdparty_refs"] = sorted(out["thirdparty_refs"])
    return out

# -----------------------------
# Gradle helpers (tags + richer detection)
# -----------------------------
AGP_PLUGIN_DSL_RX = re.compile(
    r"""id\s*\(?\s*      # id(
        [\"']com\.android\.(?:application|library|test|dynamic-feature)[\"']\s*\)?   # id("com.android.xyz")
        \s*version\s*
        [\"']([^\"']+)[\"']                      # version "X.Y.Z"
    """,
    re.I | re.X,
)

ORCHESTRATOR_COORD_RX = re.compile(r"androidx\.test:orchestrator(?::[^\s'\"\)]+)?", re.I)
ORCHESTRATOR_EXEC_RX  = re.compile(r"testOptions\s*\{[^}]*execution\s*['\"]ANDROIDX_TEST_ORCHESTRATOR['\"]", re.I | re.S)
ORCHESTRATOR_FLAG_RX  = re.compile(r"\buseOrchestrator\s*(?:=|\s)\s*true\b", re.I)
ORCHESTRATOR_PROP_RX  = re.compile(r"\bandroid(?:\.testInstrumentationRunnerArguments)?\.use(?:Test)?Orchestrator\s*=\s*true", re.I)

def extract_gradle_invocation_tags(text: str) -> Set[str]:
    return _collect_tags_from_text(text or "", TAG_PATTERNS_GRADLE)

def detect_orchestrator_from_gradle(text: str) -> bool:
    t = text or ""
    return bool(
        ORCHESTRATOR_COORD_RX.search(t) or
        ORCHESTRATOR_EXEC_RX.search(t)  or
        ORCHESTRATOR_FLAG_RX.search(t)  or
        ORCHESTRATOR_PROP_RX.search(t)
    )

# -----------------------------
# Single-file extractor (YAML + Gradle)
# -----------------------------
def extract_from_text(path: str, text: str, gradle_context_texts: Optional[List[str]] = None) -> Dict[str, Any]:
    data: Dict[str, Any] = {}

    # YAML configs (with gradle context for richer classification)
    if path.lower().endswith((".yml", ".yaml")) and yaml is not None:
        data = extract_from_yaml_text(text, gradle_context_texts or [])

    # Gradle heuristics
    if path.endswith(("build.gradle", "build.gradle.kts",
                      "gradle.properties", "gradle/wrapper/gradle-wrapper.properties",
                      "settings.gradle", "settings.gradle.kts")):
        # AGP via dependency coordinates (classpath or anywhere)
        dep_agp = re.findall(r"com\.android\.tools\.build:gradle:([0-9][^'\"\s\)]+)", text)
        # AGP via plugins { id("com.android.application") version "X" }
        dsl_agp = AGP_PLUGIN_DSL_RX.findall(text)
        agp_all = sorted(set(dep_agp + dsl_agp))
        if agp_all:
            data["agp_versions"] = agp_all

        # apiLevel = N (managed devices DSL or custom config)
        for m in re.finditer(r"\bapiLevel\s*=\s*(\d+)", text, re.IGNORECASE):
            lvl = int(m.group(1))
            data.setdefault("api_levels", [])
            if lvl not in data["api_levels"]:
                data["api_levels"].append(lvl)

        # Orchestrator signals
        if "ANDROIDX_TEST_ORCHESTRATOR" in text or detect_orchestrator_from_gradle(text):
            data["orchestrator"] = True

        # Recognize GMD via Gradle DSL (prefer gmd if present)
        if GMD_GRADLE_RX.search(text):
            current = data.get("invocation_style")
            if current in (None, "", "unknown", "diy", "gradle_connected"):
                data["invocation_style"] = "gmd"

        # Invocation tags from Gradle DSL / properties
        gtags = extract_gradle_invocation_tags(text)
        if gtags:
            data.setdefault("invocation_tags", [])
            merged = sorted(set(data["invocation_tags"]) | gtags)
            data["invocation_tags"] = merged

        # Gradle wrapper version (optional but handy)
        if path.endswith("gradle/wrapper/gradle-wrapper.properties"):
            m = re.search(r"distributionUrl=.*?/gradle-([0-9][\w\.\-]+)-", text)
            if m:
                data["gradle_wrapper_version_raw"] = m.group(1)

    return data

# -----------------------------
# IO helpers
# -----------------------------
def write_jsonl(path: Path, rows: List[dict]):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

# -----------------------------
# Main mining loop
# -----------------------------
def main():
    repos = [p for p in CLONE_ROOT.iterdir() if (p / ".git").exists()]
    repos.sort(key=lambda p: p.name.lower())
    print(f"Found {len(repos)} repos in {CLONE_ROOT}")

    ok_count = 0
    skip_count = 0
    err_count = 0

    for repo in repos:
        try:
            rel_commits = list_relevant_commits(repo)
            if MAX_COMMITS_PER_REPO > 0:
                rel_commits = rel_commits[:MAX_COMMITS_PER_REPO]

            out_rows: List[dict] = []
            for sha, ts, changed_paths in rel_commits:
                # keep only the relevant paths from that commit
                rel_paths = [p for p in changed_paths if is_ci_file(p) or is_gradle_file(p) or is_script_file(p)]
                if not rel_paths:
                    continue

                # Load all relevant file texts for this commit first (to provide Gradle context to YAML)
                path_texts: Dict[str, Optional[str]] = {}
                for pth in rel_paths:
                    path_texts[pth] = git_show(repo, sha, pth)

                # Gather Gradle texts for this commit
                gradle_texts = [txt for pth, txt in path_texts.items() if txt is not None and is_gradle_file(pth)]

                subj = git_subject(repo, sha)
                for pth in rel_paths:
                    txt = path_texts.get(pth)
                    if txt is None:
                        continue
                    feats = extract_from_text(pth, txt, gradle_context_texts=gradle_texts)
                    out_rows.append({
                        "repo": repo.name,
                        "sha": sha,
                        "timestamp": ts,
                        "subject": subj,
                        "path": pth,
                        "features": feats,
                    })

            if not out_rows:
                print(f"[skip] {repo.name}: no relevant snapshots")
                skip_count += 1
                continue

            dst = SNAPSHOT_DIR / f"{repo.name}.jsonl"
            write_jsonl(dst, out_rows)
            print(f"[ok] {repo.name}: {len(out_rows)} snapshots -> {dst}")
            ok_count += 1

        except Exception as e:
            # Never crash the whole batch; log and continue
            print(f"[err] {repo.name}: {e}")
            err_count += 1

    print(f"\nDone. ok={ok_count}, skip={skip_count}, err={err_count}, out_dir={SNAPSHOT_DIR}")

if __name__ == "__main__":
    main()


[cutoff] Using commit cutoff <= 2025-08-10 23:59:59 -0400 (epoch=1754884799)
Found 282 repos in C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones
[ok] 4eRTuk__audioview: 78 snapshots -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshots\4eRTuk__audioview.jsonl
[ok] a-mabe__OpenHIIT: 171 snapshots -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshots\a-mabe__OpenHIIT.jsonl
[ok] a914-gowtham__compose-ratingbar: 159 snapshots -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshots\a914-gowtham__compose-ratingbar.jsonl
[ok] AAkira__ExpandableLayout: 56 snapshots -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshots\AAkira__ExpandableLayout.jsonl
[ok] abdelaziz-mahdy__pytorch_lite: 131 snapshots -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshots\abdelaziz-mahdy__pytorch_lite.jsonl
[ok] ably__ably-flutter: 316 snapshots -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshots\ably__abl

In [ ]:
# step2_labeling.py
# RQ2 — Step 2, V4.5: simple subject regex; arbitration to 7 buckets; requested naming
# Emits per-source intents (both legacy CamelCase & canonical snake_case):
#   intent_Delta / intent_delta
#   intent_subject / intent_subject
#   intent_Path / intent_path
#   intent_Path_field_aware / intent_pfa
#
# Aggregation (final plan):
#   Secondary_Label  -> intents from the chosen (highest-priority) source, comma-joined
#   Driver           -> one of 7 buckets (coverage_change, flake_mitigation, speed_up_ci,
#                       version_change, orchestrator_change, invocation_change, ci_platform_change)
#   Driver_Source    -> which source won: Delta | PathFA | Subject | Path
#   Driver_reason    -> short human reason (auditable)
#
# Primary labels unchanged. Driver fields no longer "none": they reflect final intention.

from __future__ import annotations
import json, re
import datetime as _dt
from pathlib import Path
from typing import Dict, Any, List, Tuple, Optional

# -----------------------------
# Config
# -----------------------------
WORK_ROOT         = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2")
SNAPSHOT_DIR      = WORK_ROOT / "snapshots"
CCE_ENRICHED_DIR  = WORK_ROOT / "cce_enriched_V4.5"
CCE_ENRICHED_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Small helpers
# -----------------------------
VERSION_RE = re.compile(r"\d+(?:\.\d+)*")

def read_snapshots(folder: Path) -> Dict[str, List[dict]]:
    by_repo: Dict[str, List[dict]] = {}
    for p in folder.glob("*.jsonl"):
        with p.open(encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                d = json.loads(line)
                repo = d.get("repo")
                if not repo:
                    continue
                by_repo.setdefault(repo, []).append(d)
    # chronological per path
    for repo, rows in by_repo.items():
        rows.sort(key=lambda r: (r.get("path",""), int(r.get("timestamp", 0)), r.get("sha","")))
    return by_repo

def as_set_str(xs) -> set:
    if xs is None: return set()
    if isinstance(xs, (list, set, tuple)):
        return set(str(x) for x in xs)
    return {str(xs)}

def as_set_int(xs) -> set:
    if xs is None: return set()
    out = set()
    if isinstance(xs, (list, set, tuple)):
        for x in xs:
            try: out.add(int(x))
            except: pass
    else:
        try: out.add(int(xs))
        except: pass
    return out

def parse_version_tuple(s: str) -> Tuple[int, ...]:
    m = VERSION_RE.search(str(s) if s is not None else "")
    if not m: return tuple()
    parts = m.group(0).split(".")
    out: List[int] = []
    for p in parts:
        try: out.append(int(p))
        except: out.append(0)
    return tuple(out)

def max_version_tuple(strings: List[str]) -> Tuple[int, ...]:
    best: Tuple[int, ...] = tuple()
    for s in strings or []:
        vt = parse_version_tuple(str(s))
        if vt > best: best = vt
    return best

def stringify(x: Any) -> str:
    if isinstance(x, (dict, list, set, tuple)):
        try: return json.dumps(x, ensure_ascii=False, sort_keys=True)
        except: return str(x)
    return "" if x is None else str(x)

def _safe_json_loads(s: str):
    try: return json.loads(s) if s else None
    except: return None

# -----------------------------
# Primary label mapping (WHAT)
# -----------------------------
FIELD_TO_PRIMARY = {
    "api_levels":        "api_bump",
    "agp_versions":      "agp_bump",
    "jdk":               "jdk_bump",
    "runner_os":         "runner_os_change",
    "matrix_axes":       "matrix_change",
    "orchestrator":      "orchestrator_change",
    "timeouts":          "timeout_tuning",
    "retries":           "retry_tuning",
    "device_profiles":   "device_profile_change",
    "abis":              "abi_change",
    "system_images":     "system_image_change",
    "invocation_style":  "invocation_change",
    "invocation_tags":   "invocation_change",
    "thirdparty_refs":   "external_service_change",
    "wait_for_boot":     "wait_strategy_change",
}

# 4 categories (IDs + pretty names)
CAT_RUNTIME  = "runtime"      # Runtime Resilience & Throughput Tuning
CAT_COVERAGE = "coverage"     # Test Surface & Coverage Configuration
CAT_CI       = "ci_platform"  # CI Platform & Integrations
CAT_TOOL     = "toolchain"    # Toolchain & Invocation Evolution

CATEGORY_NAME = {
    CAT_RUNTIME:  "Runtime Resilience & Throughput Tuning",
    CAT_COVERAGE: "Test Surface & Coverage Configuration",
    CAT_CI:       "CI Platform & Integrations",
    CAT_TOOL:     "Toolchain & Invocation Evolution",
}

PRIMARY_TO_CATEGORY = {
    "timeout_tuning":         CAT_RUNTIME,
    "retry_tuning":           CAT_RUNTIME,
    "orchestrator_change":    CAT_RUNTIME,
    "wait_strategy_change":   CAT_RUNTIME,

    "api_bump":               CAT_COVERAGE,
    "abi_change":             CAT_COVERAGE,
    "device_profile_change":  CAT_COVERAGE,
    "system_image_change":    CAT_COVERAGE,
    "matrix_change":          CAT_COVERAGE,

    "runner_os_change":       CAT_CI,
    "external_service_change":CAT_CI,

    "invocation_change":      CAT_TOOL,
    "agp_bump":               CAT_TOOL,
    "jdk_bump":               CAT_TOOL,

    "other_change":           CAT_CI,  # conservative default if unmapped
}

# -----------------------------
# Diff engine → per-field CCEs
# -----------------------------
def diff_features(old: Dict[str, Any], new: Dict[str, Any]) -> List[Dict[str, Any]]:
    old = old or {}; new = new or {}
    out: List[Dict[str, Any]] = []

    def handle_set(field: str, to_set_fn):
        a = to_set_fn(old.get(field)); b = to_set_fn(new.get(field))
        if a == b: return
        added   = sorted(b - a)
        removed = sorted(a - b)
        row = {
            "field": field,
            "old_value": stringify(sorted(a)),
            "new_value": stringify(sorted(b)),
            "change_type": "modified",
        }
        if added:   row["added_items"] = stringify(added)
        if removed: row["removed_items"] = stringify(removed)
        if field == "api_levels" and a and b:
            try: row["magnitude"] = max(b) - max(a)
            except: pass
        out.append(row)

    # Set-like
    handle_set("api_levels", as_set_int)
    handle_set("abis", as_set_str)
    handle_set("system_images", as_set_str)
    handle_set("device_profiles", as_set_str)
    handle_set("matrix_axes", as_set_str)
    handle_set("invocation_tags", as_set_str)
    handle_set("thirdparty_refs", as_set_str)

    # AGP versions (ordered strings → compare sets + direction)
    a_agp = sorted(as_set_str(old.get("agp_versions"))); b_agp = sorted(as_set_str(new.get("agp_versions")))
    if a_agp != b_agp:
        row = {"field":"agp_versions","old_value":stringify(a_agp),"new_value":stringify(b_agp),"change_type":"modified"}
        old_max = max_version_tuple(list(a_agp)); new_max = max_version_tuple(list(b_agp))
        if old_max or new_max:
            row["magnitude"] = 1 if new_max > old_max else (-1 if new_max < old_max else 0)
        out.append(row)

    # Scalar
    for field in ("orchestrator","wait_for_boot","retries","runner_os","jdk","invocation_style"):
        a = old.get(field, None); b = new.get(field, None)
        if a != b:
            out.append({
                "field": field, "old_value": stringify(a), "new_value": stringify(b),
                "change_type": ("modified" if (a is not None and b is not None)
                                else ("added" if a is None else "removed")),
            })

    # Dict-ish
    for field in ("timeouts",):
        a = old.get(field, None); b = new.get(field, None)
        if stringify(a) != stringify(b):
            out.append({
                "field": field, "old_value": stringify(a), "new_value": stringify(b),
                "change_type": ("modified" if (a is not None and b is not None)
                                else ("added" if a is None else "removed")),
            })
    return out

# -----------------------------
# Delta-side intents (extraction only)
# -----------------------------
_DURATION_RX = re.compile(r"(?i)^\s*(\d+(?:\.\d+)?)\s*(ms|s|m|h)?\s*$")
_UNIT_TO_SEC = {"ms": 0.001, "s": 1, "m": 60, "h": 3600}

def _to_seconds(x) -> Optional[float]:
    if x is None: return None
    if isinstance(x, (int, float)): return float(x)
    m = _DURATION_RX.match(str(x))
    if not m: return None
    val = float(m.group(1)); unit = (m.group(2) or "s").lower()
    return val * _UNIT_TO_SEC.get(unit, 1)

def _sum_timeout_seconds(obj) -> Optional[float]:
    if obj is None: return None
    total = 0.0; seen = 0
    if isinstance(obj, dict):
        it = obj.values()
    elif isinstance(obj, list):
        it = [v for _, v in obj if isinstance(_, (str,int))]
    else:
        return None
    for v in it:
        secs = _to_seconds(v)
        if secs is not None:
            total += secs; seen += 1
    return total if seen else None

def derive_delta_intents(deltas: List[Dict[str, Any]]) -> List[str]:
    """
    Extracts raw intents from feature-level diffs.
    Note: we only assert version_change when both sides are parseable (clean).
    """
    labs: set[str] = set()
    for d in deltas:
        field = d.get("field")
        old_v = d.get("old_value"); new_v = d.get("new_value")
        added = _safe_json_loads(d.get("added_items") or "") or []
        removed = _safe_json_loads(d.get("removed_items") or []) or []

        # Coverage sets
        if field in {"api_levels", "abis", "device_profiles", "system_images"}:
            if added:   labs.add("expand_coverage")
            if removed: labs.add("reduce_coverage")

        # Matrix axes
        if field == "matrix_axes":
            labs.add("coverage_dimensions_change")
            if added:   labs.add("expand_coverage")
            if removed: labs.add("reduce_coverage")

        # Timeouts
        if field == "timeouts":
            labs.add("timeout_tuning")
            old_obj = _safe_json_loads(old_v); new_obj = _safe_json_loads(new_v)
            old_s = _sum_timeout_seconds(old_obj); new_s = _sum_timeout_seconds(new_obj)
            if old_s is not None and new_s is not None:
                if new_s > old_s: labs.add("flake_mitigation")
                elif new_s < old_s: labs.add("speed_up_ci")

        # Retries
        if field == "retries":
            labs.add("retry_tuning")
            try: a = int(_safe_json_loads(old_v) if old_v else 0)
            except: a = None
            try: b = int(_safe_json_loads(new_v) if new_v else 0)
            except: b = None
            if a is not None and b is not None:
                if b > a: labs.add("flake_mitigation")
                elif b < a: labs.add("speed_up_ci")

        # Orchestrator
        if field == "orchestrator": labs.add("orchestrator_change")

        # Runner platform
        if field == "runner_os": labs.add("infra_tuning")

        # Invocation strategy flips (these are not super common; keep them visible)
        if field == "invocation_style":
            old_s = (old_v or "").strip().lower()
            new_s = (new_v or "").strip().lower()
            if new_s == "gmd" and old_s != "gmd": labs.add("adopt_gmd")
            if old_s == "gmd" and new_s != "gmd": labs.add("drop_gmd")
            if new_s == "diy" and old_s != "diy": labs.add("adopt_diy")

        # External services
        if field == "thirdparty_refs": labs.add("external_service_change")

        # >>> NEW: strict version_change via Delta (only when both parse cleanly)
        if field in {"agp_versions", "jdk"}:
            a = _safe_json_loads(old_v) if old_v else None
            b = _safe_json_loads(new_v) if new_v else None
            # old/new may be list (agp) or scalar (jdk)
            if field == "agp_versions":
                a_list = a if isinstance(a, list) else (list(as_set_str(a)) if a else [])
                b_list = b if isinstance(b, list) else (list(as_set_str(b)) if b else [])
                a_max = max_version_tuple([str(x) for x in a_list])
                b_max = max_version_tuple([str(x) for x in b_list])
                if a_max and b_max and a_max != b_max:
                    labs.add("version_change")
            elif field == "jdk":
                a_v = parse_version_tuple(a if isinstance(a, str) else (str(a) if a is not None else ""))
                b_v = parse_version_tuple(b if isinstance(b, str) else (str(b) if b is not None else ""))
                if a_v and b_v and a_v != b_v:
                    labs.add("version_change")
    return sorted(labs)

# -----------------------------
# Path detectors (generic + field-aware)
# -----------------------------
CI_VENDOR_PATTERNS = [
    (re.compile(r'(?i)(?:^|/)\.travis\.ya?ml$'),               'Travis_CI'),
    (re.compile(r'(?i)(?:^|/)\.appveyor\.ya?ml$'),             'AppVeyor'),
    (re.compile(r'(?i)(?:^|/)appveyor\.ya?ml$'),               'AppVeyor'),
    (re.compile(r'(?i)(?:^|/)circle\.yml$'),                   'Circle_CI'),
    (re.compile(r'(?i)(?:^|/)\.circleci/config\.ya?ml$'),      'Circle_CI'),
    (re.compile(r'(?i)(?:^|/)azure-pipelines\.ya?ml$'),        'Azure_Pipelines'),
    (re.compile(r'(?i)(?:^|/)\.github/workflows/.*\.(yml|yaml)$'), 'GitHub_Actions'),
    (re.compile(r'(?i)(?:^|/)bitbucket-pipelines\.ya?ml$'),    'Bitbucket'),
    (re.compile(r'(?i)(?:^|/)\.gitlab-ci\.ya?ml$'),            'GitLab'),
    (re.compile(r'(?i)(?:^|/)(?:Jenkinsfile(?:\.ya?ml)?)$'),   'Jenkins'),
    (re.compile(r'(?i)(?:^|/)bitrise\.ya?ml$'),                'Bitrise'),
    (re.compile(r'(?i)(?:^|/)bamboo\.ya?ml$'),                 'Bamboo'),
    (re.compile(r'(?i)(?:^|/)codeship-services\.ya?ml$'),      'Codeship'),
    (re.compile(r'(?i)(?:^|/)\.gocd\.ya?ml$'),                 'GoCD'),
    (re.compile(r'(?i)(?:^|/)\.cirrus\.ya?ml$'),               'Cirrus'),
    (re.compile(r'(?i)(?:^|/)(?:wercker|semaphore)\.ya?ml$'),  'Semaphore'),
    (re.compile(r'(?i)(?:^|/)codemagic\.ya?ml$'),              'Nevercode'),
]
GENERIC_CI_DIRS   = re.compile(r'(?i)(?:^|/)(?:ci|\.ci)(?:/|$)')
_GRADLE_PATH_RX   = re.compile(r"(?i)(?:^|/)(?:build|settings)\.gradle(?:\.kts)?$|(?:^|/)gradle\.properties$|(?:^|/)gradle/wrapper/gradle-wrapper\.properties$")
_SCRIPT_PATH_RX   = re.compile(r"(?i)(?:^|/)(?:scripts?|tools?)(?:/|$)|\.(?:sh|py|bat|ps1)$")

_COVERAGE_FIELDS = {"api_levels", "abis", "device_profiles", "system_images", "matrix_axes"}

def detect_intent_path_generic(path: str) -> List[str]:
    """Path-only (no field knowledge)."""
    intents: List[str] = []
    p = path or ""
    if any(rx.search(p) for rx, _ in CI_VENDOR_PATTERNS) or GENERIC_CI_DIRS.search(p):
        intents.append("ci_env_workflow")
    if _GRADLE_PATH_RX.search(p):
        intents.append("version_change")
    if _SCRIPT_PATH_RX.search(p):
        intents.append("ci_env_workflow")
    return sorted(set(intents))

def detect_intent_path_field_aware(path: str, field: str) -> Tuple[List[str], Optional[str]]:
    """Field-aware path intents + short reason."""
    p = path or ""; f = (field or "").lower()
    intents: List[str] = []
    reason: Optional[str] = None

    vendor = None
    for rx, name in CI_VENDOR_PATTERNS:
        if rx.search(p):
            vendor = name; break
    if vendor is None and GENERIC_CI_DIRS.search(p):
        vendor = "Generic_CI"

    if vendor:
        if f in _COVERAGE_FIELDS:
            intents += ["coverage", "ci_env_workflow"]  # coverage wins later in bucket ranking
            reason = f"ci vendor={vendor} + coverage field"
        elif f in {"runner_os", "thirdparty_refs"}:
            intents += ["ci_env_workflow"]
            reason = f"ci vendor={vendor} + {f}"
        else:
            intents += ["ci_env_workflow"]
            reason = f"ci vendor={vendor}"

    if _GRADLE_PATH_RX.search(p):
        if f in {"agp_versions", "jdk"}:
            intents.append("version_change")
            reason = (reason or "") + ("; " if reason else "") + "gradle/build tool path"

    if _SCRIPT_PATH_RX.search(p):
        intents.append("ci_env_workflow")
        reason = (reason or "") + ("; " if reason else "") + "script/tool path"

    intents = sorted(set(intents))
    return intents, (reason or None)

# -----------------------------
# Subject-driven intents (simple regex, as provided)
# -----------------------------
MIGRATE_VERBS  = r"(?:migrat|switch|move(?:\s*to)?|replace|port)"
CI_VENDORS_TXT = r"(?:github actions|gha|gitlab|jenkins|circleci|azure pipelines|bitrise)"
TOOLS          = r"(?:agp|gradle(?:\s*wrapper)?|jdk|java|toolchain|target\s*api)"
VERSION        = r"(?:\d+(?:\.\d+)+|1[1-9]|[89])"

# stability
FIX_OR_HOTFIX = r"(?:fix(?:es|ed)?|hotfix|unblock)"
CRASH_TERMS   = r"(?:crash(?:es|ing)?)"
FLAKE_TERMS   = r"(?:de[-_ ]?flake\w*|flak(?:e|y|iness))"
STAB_TERMS    = r"(?:stabilize|stability|intermittent)"
TIME_RETRY    = r"(?:timeout|timeouts?|retry|retries)"
SIGNALS       = rf"(?:{FLAKE_TERMS}|{STAB_TERMS}|{TIME_RETRY}|{CRASH_TERMS})"

# CI speed / throughput
SPEED_TERMS    = r"(?:speed(?:\s*-?\s*)?up|faster|reduce\s*time|time\s*to\s*green|parallel|shard(?:ing)?|concurr\w*)"
CI_CONTEXT     = r"(?:ci|build|pipeline|workflow|runner)"
TEST_CONTEXT   = r"(?:test|tests)"

REVERT_TERMS   = r"(?:revert|roll\s*back|back\s*out)"
CLEANUP_TERMS  = r"(?:clean\s*up|tidy|refactor|format|lint)"

RE_CI_VENDOR = re.compile(
    rf"(?i)(?:\b{MIGRATE_VERBS}\b.{{0,40}}\b{CI_VENDORS_TXT}\b|\b{CI_VENDORS_TXT}\b.{{0,40}}\b{MIGRATE_VERBS}\b)"
)
RE_VERSION_CHG = re.compile(
    rf"(?i)(?:\b(?:bump|upgrad\w*|updat\w*|pin|unpin)\b.{{0,24}}\b{TOOLS}\b|"
    rf"\b{TOOLS}\b.{{0,24}}\b{VERSION}\b|"
    rf"\bversion\s+{VERSION}(?:\s*(?:->|=>|>)\s*{VERSION})?)"
)

RE_STABILITY = re.compile(rf"(?i)\b{SIGNALS}\b")
RE_SPEED_CI  = re.compile(rf"(?i)(?:\b{SPEED_TERMS}\b.{{0,20}}\b{CI_CONTEXT}\b|\b{CI_CONTEXT}\b.{{0,20}}\b{SPEED_TERMS}\b)")
RE_REVERT    = re.compile(rf"(?i)\b{REVERT_TERMS}\b")
RE_CLEANUP   = re.compile(rf"(?i)\b{CLEANUP_TERMS}\b")

def detect_subject_intents(subject: str) -> Tuple[List[str], List[str], Optional[str]]:
    s = subject or ""
    intents: List[str] = []
    aux: List[str] = []
    reason: Optional[str] = None

    # Aux tags (record only)
    if RE_REVERT.search(s):  aux.append("revert")
    if RE_CLEANUP.search(s): aux.append("cleanup_refactor")
    if re.search(r"(?i)\b(docs?|readme|changelog)\b", s): aux.append("docs_changelog")
    if re.search(r"(?i)\b(release|tag(?:ging)?|rc|beta|alpha|nightly)\b", s): aux.append("release_tagging")
    if re.search(r"(?i)\b(remove|delete|drop|deprecat\w+)\b", s): aux.append("removal_deprecation")

    # Intents (subject-only; no arbitration)
    if RE_STABILITY.search(s):
        intents.append("flake_mitigation"); reason = reason or "stability"
    if RE_SPEED_CI.search(s):
        intents.append("speed_up_ci");      reason = reason or "speed/ci"
    if RE_CI_VENDOR.search(s):
        intents.append("ci_env_workflow");  reason = reason or "ci_vendor"
    if RE_VERSION_CHG.search(s):
        intents.append("version_change");   reason = reason or "version_change"

    intents = sorted(set(intents))
    aux     = sorted(set(aux))
    return intents, aux, reason

# -----------------------------
# Tagged secondary labels helper (legacy)
# -----------------------------
def make_tagged_secondary(intent_Delta: List[str], intent_subject: List[str],
                          intent_Path: List[str], intent_Path_field_aware: List[str]) -> List[str]:
    tagged = []
    for t in sorted(set(intent_Delta)):            tagged.append(f"1_{t}")
    for t in sorted(set(intent_subject)):          tagged.append(f"2_{t}")
    for t in sorted(set(intent_Path)):             tagged.append(f"3_{t}")
    for t in sorted(set(intent_Path_field_aware)): tagged.append(f"3f_{t}")
    # de-dup preserving order
    seen = set(); out = []
    for x in tagged:
        if x not in seen:
            seen.add(x); out.append(x)
    return out

# -----------------------------
# Change-op (compact)
# -----------------------------
def classify_change_op(old_value: str, new_value: str, change_type: str) -> str:
    ct = (change_type or "").lower()
    if ct == "added":   return "add"
    if ct == "removed": return "remove"
    return "value_edit" if (old_value or "") != (new_value or "") else "no_change"

# -----------------------------
# UTC timestamp helpers
# -----------------------------
def ensure_utc_epoch(ts_any) -> int:
    try: ts = int(float(ts_any))
    except: ts = 0
    return ts

def epoch_to_iso_utc(ts: int) -> str:
    return _dt.datetime.fromtimestamp(int(ts), tz=_dt.timezone.utc).isoformat().replace("+00:00", "Z")

# -----------------------------
# >>> NEW: Arbitration to 7 buckets
# -----------------------------
# Buckets priority (within chosen source)
BUCKET_ORDER = [
    "coverage_change",
    "flake_mitigation",
    "speed_up_ci",
    "version_change",
    "orchestrator_change",
    "invocation_change",
    "ci_platform_change",
]

RAW_TO_BUCKET = {
    # coverage
    "coverage_dimensions_change":"coverage_change",
    "expand_coverage":"coverage_change",
    "reduce_coverage":"coverage_change",
    "coverage":"coverage_change",  # from Path-FA
    # runtime
    "flake_mitigation":"flake_mitigation",
    "timeout_tuning":"flake_mitigation",
    "retry_tuning":"flake_mitigation",
    "speed_up_ci":"speed_up_ci",
    "orchestrator_change":"orchestrator_change",
    # toolchain
    "version_change":"version_change",
    "adopt_gmd":"invocation_change",
    "drop_gmd":"invocation_change",
    "adopt_diy":"invocation_change",
    # ci platform
    "ci_env_workflow":"ci_platform_change",
    "infra_tuning":"ci_platform_change",
    "external_service_change":"ci_platform_change",
}

def _to_set(s: Any) -> set:
    if s is None: return set()
    if isinstance(s, list): return set(str(x) for x in s)
    if isinstance(s, str):
        s = s.strip()
        if not s: return set()
        try:
            v = json.loads(s)
            if isinstance(v, list):
                return set(str(x) for x in v)
        except: pass
        return set(t.strip() for t in s.split(",") if t.strip())
    return {str(s)}

def map_raw_to_buckets(raws: List[str] | set) -> set:
    return {RAW_TO_BUCKET[r] for r in (list(raws) if isinstance(raws,set) else raws) if r in RAW_TO_BUCKET}

def pick_bucket(bset: set) -> str:
    for b in BUCKET_ORDER:
        if b in bset: return b
    return ""

def build_driver_reason(driver: str, src: str, row: dict) -> str:
    ipr = row.get("Intent_Path_Reason") or ""
    isr = row.get("Intent_Subject_Reason") or ""
    if src == "Delta":
        if driver == "coverage_change":     return "coverage set diff in Delta (added/removed)"
        if driver == "flake_mitigation":    return "timeouts/retries increased (Delta)"
        if driver == "speed_up_ci":         return "timeouts/retries decreased (Delta)"
        if driver == "version_change":      return "clean version bump parsed (old→new)"
        if driver == "orchestrator_change": return "orchestrator flipped (Delta)"
        if driver == "ci_platform_change":  return "runner_os/third-party changed (Delta)"
        if driver == "invocation_change":   return "invocation_style change (Delta/subject/path)"
    if src == "PathFA":
        if driver == "coverage_change":     return f"CI config + coverage field ({ipr})"
        if driver == "version_change":      return f"Gradle/AGP/JDK path evidence ({ipr})"
        if driver == "ci_platform_change":  return f"CI workflow path ({ipr})"
        return ipr or "field-aware path"
    if src == "Subject":
        return isr or "subject regex"
    if src == "Path":
        return ipr or "path heuristic"
    return "no signal"

# -----------------------------
# Main
# -----------------------------
if __name__ == "__main__":
    print(f"[info] snapshots dir: {SNAPSHOT_DIR}")
    print(f"[info] output dir   : {CCE_ENRICHED_DIR}")

    by_repo = read_snapshots(SNAPSHOT_DIR)
    print(f"[info] Loaded snapshots for {len(by_repo)} repos")

    for repo, rows in by_repo.items():
        out_rows: List[dict] = []
        by_path: Dict[str, List[dict]] = {}
        for r in rows:
            by_path.setdefault(r.get("path",""), []).append(r)

        # Track repeats per (path, field) within the repo
        repeat_counter: Dict[Tuple[str,str], int] = {}

        for path, snaps in by_path.items():
            prev: Optional[dict] = None
            for cur in snaps:
                if prev is None:
                    prev = cur
                    continue

                old_feats = prev.get("features", {}) or {}
                new_feats = cur.get("features", {}) or {}
                deltas = diff_features(old_feats, new_feats)

                if deltas:
                    # Episode-level detection shared across CCEs in this episode
                    intent_Delta_all = derive_delta_intents(deltas)
                    subject_str = cur.get("subject","")
                    intent_subject_all, aux_subject, intent_subject_reason_global = detect_subject_intents(subject_str)

                    # UTC timestamps
                    ts_epoch_utc = ensure_utc_epoch(cur.get("timestamp", 0))
                    ts_iso_utc   = epoch_to_iso_utc(ts_epoch_utc)

                    for d in deltas:
                        field = d["field"]
                        key = (path, field)
                        repeat_counter[key] = repeat_counter.get(key, 0) + 1
                        repeat_index = repeat_counter[key]
                        repeat_label = "first" if repeat_index == 1 else "repeat"

                        # Primary WHAT (exactly one per CCE)
                        primary_label = FIELD_TO_PRIMARY.get(field, "other_change")
                        primary_category_id = PRIMARY_TO_CATEGORY.get(primary_label, CAT_CI)
                        primary_category = CATEGORY_NAME.get(primary_category_id, primary_category_id)

                        # Path intents (generic + field-aware, per CCE)
                        intent_Path      = detect_intent_path_generic(path)
                        intent_Path_fa, intent_Path_reason = detect_intent_path_field_aware(path, field)

                        # Per-source intents (independent; no arbitration)
                        intent_Delta   = list(sorted(set(intent_Delta_all)))
                        intent_subject = list(sorted(set(intent_subject_all)))

                        # >>> NEW: Arbitration (Strategy A) to produce Secondary_Label + Driver
                        # Build pools
                        pools_raw = {
                            "Delta":   set(intent_Delta),
                            "PathFA":  set(intent_Path_fa),
                            "Subject": set(intent_subject),
                            "Path":    set(intent_Path),
                        }

                        # choose highest-priority non-empty source
                        chosen_src = None
                        for s in ("Delta","PathFA","Subject","Path"):
                            if pools_raw[s]:
                                chosen_src = s
                                break

                        if chosen_src:
                            sec_list = sorted(pools_raw[chosen_src])  # stable order
                            secondary_label_str = ",".join(sec_list)
                            bucket_set = map_raw_to_buckets(pools_raw[chosen_src])
                            driver = pick_bucket(bucket_set)
                            driver_source = chosen_src
                            driver_reason = build_driver_reason(driver, driver_source, {
                                "Intent_Path_Reason": intent_Path_reason,
                                "Intent_Subject_Reason": intent_subject_reason_global
                            })
                        else:
                            secondary_label_str = ""
                            driver = ""
                            driver_source = None
                            driver_reason = "no signal"

                        # Aggregations (legacy, preserved)
                        secondary_union = sorted(set(intent_Delta) | set(intent_subject) | set(intent_Path) | set(intent_Path_fa))
                        secondary_tagged = make_tagged_secondary(intent_Delta, intent_subject, intent_Path, intent_Path_fa)

                        # change_op
                        change_op = classify_change_op(d.get("old_value",""), d.get("new_value",""), d.get("change_type","modified"))

                        # Row out
                        row_out = {
                            # --- General context
                            "repo": repo,
                            "sha": cur.get("sha"),
                            "prev_sha": prev.get("sha"),
                            "timestamp_epoch_utc": ts_epoch_utc,
                            "timestamp_utc": ts_iso_utc,
                            "path": path,
                            "subject": subject_str,

                            # --- Primary (WHAT)
                            "Field": field,
                            "Primary_Label": primary_label,
                            "Primary_Label_Category": primary_category,

                            # --- Delta details
                            "old_value": d.get("old_value",""),
                            "new_value": d.get("new_value",""),
                            "change_type": d.get("change_type","modified"),
                            "change_op": change_op,
                            "magnitude": d.get("magnitude", None),
                            "added_items": d.get("added_items",""),
                            "removed_items": d.get("removed_items",""),
                            "repeat_index": repeat_index,
                            "repeat_label": repeat_label,

                            # --- Per-source intents (legacy naming)
                            "intent_Delta": intent_Delta,
                            "intent_subject": intent_subject,
                            "intent_Path": intent_Path,
                            "intent_Path_field_aware": intent_Path_fa,

                            # --- Canonical per-source intents (snake_case; same content)
                            "intent_delta": ",".join(intent_Delta),
                            "intent_subject_snake": ",".join(intent_subject),  # keep name distinct from legacy col
                            "intent_path": ",".join(intent_Path),
                            "intent_pfa": ",".join(intent_Path_fa),

                            # --- Aggregated (legacy/compat)
                            "Secondary_Label": secondary_label_str,              # chosen source only (final plan)
                            "Secondary_Label_Tagged": secondary_tagged,         # legacy tagged (1_/2_/3_/3f_)
                            "Secondary_Label_Tagged_Text": ", ".join(secondary_tagged),

                            # --- Aux & audit
                            "Aux_Tags": aux_subject,
                            "Intent_Path": "; ".join(intent_Path_fa) if intent_Path_fa else None,  # legacy compat
                            "Intent_Path_Reason": intent_Path_reason,
                            "Intent_Subject_Reason": intent_subject_reason_global,

                            # --- Driver (final 7-bucket intention)
                            "Driver": driver,
                            "Driver_Source": driver_source,
                            "Driver_Label_Selected": driver,  # kept for compat with your Step-3 expectations
                            "Driver_reason": driver_reason,
                        }

                        out_rows.append(row_out)

                prev = cur

        if not out_rows:
            print(f"[skip] {repo}: no field-level deltas found")
            continue

        dst = CCE_ENRICHED_DIR / f"{repo}.jsonl"
        with dst.open("w", encoding="utf-8") as f:
            for r in out_rows:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        print(f"[ok] {repo}: {len(out_rows)} enriched rows -> {dst}")

    print("Done.")


[info] snapshots dir: C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshots
[info] output dir   : C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\cce_enriched_V4.4
[info] Loaded snapshots for 282 repos
[ok] 4eRTuk__audioview: 10 enriched rows -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\cce_enriched_V4.4\4eRTuk__audioview.jsonl
[ok] a-mabe__OpenHIIT: 16 enriched rows -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\cce_enriched_V4.4\a-mabe__OpenHIIT.jsonl
[ok] a914-gowtham__compose-ratingbar: 18 enriched rows -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\cce_enriched_V4.4\a914-gowtham__compose-ratingbar.jsonl
[ok] AAkira__ExpandableLayout: 5 enriched rows -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\cce_enriched_V4.4\AAkira__ExpandableLayout.jsonl
[ok] abdelaziz-mahdy__pytorch_lite: 24 enriched rows -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\cce_enriched_V4.4\abdelaziz-mahdy__pytorch_lite.json

In [ ]:
# %% [markdown]
# RQ2 — Step 3 (Combine V4.5): Snapshots + Enriched Episodes
# - Snapshots: keep `features_json` as a string (drop raw `features`)
# - Episodes (from Step 2 V4.5+):
#     * Keep ONLY columns produced by Step 2 (plus _source_file)
#     * Column order:
#         1) Metadata
#         2) Diff/audit/misc
#         3) Primary (Field, Primary_Label, Primary_Label_Category)
#         4) Driver (Driver, Driver_Source, Driver_Label_Selected, Driver_reason)
#         5) Secondary labels (union + tagged + per-source intents + Aux + reasons)
#     * If 'change_op' missing (older Step-2), compute conservative value from old/new.
#     * Ensure **no [] in any column**: list-like values -> deduped, sorted, comma-separated.
# - Parquet: optional, with safe dtypes for the slim schema

from __future__ import annotations
import json, csv, os, re, ast, math
from pathlib import Path
from typing import List, Any, Optional, Tuple

# -----------------------------
# Config (updated to V4.5)
# -----------------------------
WORK_ROOT         = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2")
SNAPSHOT_DIR      = WORK_ROOT / "snapshots"             # Step 1 output

# Prefer newest Step-2 output; keep older as fallback for compatibility
CCE_ENRICHED_DIRS = [
    WORK_ROOT / "cce_enriched_V4.5",   # Step-2 output (Label+ V4.5 with Driver fields)
    WORK_ROOT / "cce_enriched_V4.4",
    WORK_ROOT / "cce_enriched_V4.2",
]
CCE_ENRICHED_DIR = next((p for p in CCE_ENRICHED_DIRS if p.exists()), CCE_ENRICHED_DIRS[0])

COMBINE_DIR       = WORK_ROOT / "combined_V4.5"
COMBINE_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Helpers
# -----------------------------
def read_all_jsonl(folder: Path) -> List[dict]:
    out: List[dict] = []
    if not folder.exists():
        return out
    for p in folder.glob("*.jsonl"):
        with p.open(encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    d = json.loads(line)
                    d["_source_file"] = p.name
                    out.append(d)
    return out

def to_json(x: Any) -> str:
    try:
        return json.dumps(x, ensure_ascii=False, sort_keys=True)
    except Exception:
        return "" if x is None else str(x)

def _is_na_like(x: Any) -> bool:
    if x is None:
        return True
    if isinstance(x, float):
        try:
            return math.isnan(x)
        except Exception:
            return False
    if isinstance(x, str):
        s = x.strip().lower()
        return s in {"", "null", "none", "nan", "na"}
    return False

def conservative_change_op(old_v: Any, new_v: Any) -> str:
    """Fallback if Step-2 didn't include 'change_op'."""
    o_empty = _is_na_like(old_v)
    n_empty = _is_na_like(new_v)
    if o_empty and n_empty:
        return "no_change"
    if o_empty and not n_empty:
        return "add"
    if not o_empty and n_empty:
        return "remove"
    return "value_edit" if (str(old_v) != str(new_v)) else "no_change"

# ---- list-like -> comma string (no brackets), deduped & sorted ----
def to_list_like(x: Any) -> list:
    """Return a best-effort list for list-like inputs."""
    if x is None:
        return []
    if isinstance(x, list):
        return x
    if isinstance(x, (set, tuple)):
        return list(x)
    if isinstance(x, str):
        s = x.strip()
        if not s:
            return []
        # JSON-style list
        if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
            # try JSON first
            try:
                v = json.loads(s)
                if isinstance(v, list):
                    return v
            except Exception:
                pass
            # then literal_eval for python-y list/tuple/set
            try:
                v = ast.literal_eval(s)
                if isinstance(v, (list, tuple, set)):
                    return list(v)
            except Exception:
                pass
        # simple comma/semicolon delimited fallback
        if "," in s:
            return [t.strip() for t in s.split(",") if t.strip()]
        if ";" in s:
            return [t.strip() for t in s.split(";") if t.strip()]
        # plain token -> single-item list
        return [s]
    # anything else becomes a single-item list
    return [x]

def clean_cell_no_brackets(v: Any) -> str:
    """
    Ensure a CSV-friendly cell: if list-like, dedupe+sort+comma-join.
    If string that *looks* like a list (e.g., '["a","b"]'), parse & join.
    Otherwise, return as string unchanged. Dicts remain JSON.
    """
    if v is None:
        return ""
    if isinstance(v, dict):
        try:
            return json.dumps(v, ensure_ascii=False, sort_keys=True)
        except Exception:
            return str(v)

    if isinstance(v, (list, tuple, set)) or (isinstance(v, str) and v.strip()[:1] in "[(" and v.strip()[-1:] in "])"):
        items = [str(t).strip() for t in to_list_like(v) if str(t).strip()]
        items = sorted(set(items))
        return ",".join(items)

    s = str(v)
    st = s.strip()
    if (st.startswith("[") and st.endswith("]")) or (st.startswith("(") and st.endswith(")")):
        items = [str(t).strip() for t in to_list_like(st) if str(t).strip()]
        items = sorted(set(items))
        return ",".join(items)
    return s

# --- Name normalization (V4.2/V4.4 → V4.5) ---
def normalize_episode_row_names(r: dict) -> dict:
    """
    Map older outputs to the V4.5 schema so downstream code always
    writes the new column names.
    """
    rr = dict(r)

    # -------- Secondary_Label (single-source summary) --------
    # V4.2 used "Secondary_Labels" (union); prefer explicit Secondary_Label if present.
    if "Secondary_Label" not in rr and "Secondary_Labels" in rr:
        rr["Secondary_Label"] = rr.get("Secondary_Labels", "")

    # -------- Per-source intents --------
    # Legacy camelCase lists exist in all versions, keep as-is.
    # Canonical snake_case (strings) are new in V4.5; backfill if needed.
    # intent_delta
    if "intent_delta" not in rr:
        val = rr.get("intent_Delta", "")
        rr["intent_delta"] = clean_cell_no_brackets(val)
    # intent_pfa
    if "intent_pfa" not in rr:
        val = rr.get("intent_Path_field_aware", "")
        rr["intent_pfa"] = clean_cell_no_brackets(val)
    # intent_path
    if "intent_path" not in rr:
        val = rr.get("intent_Path", "")
        rr["intent_path"] = clean_cell_no_brackets(val)
    # intent_subject (string) — prefer explicit snake if provided; else stringify legacy list
    if "intent_subject" not in rr:
        val = rr.get("intent_subject_snake", rr.get("intent_subject", ""))
        rr["intent_subject"] = clean_cell_no_brackets(val)

    # -------- Tagged forms (legacy compat) --------
    rr.setdefault("Secondary_Label_Tagged", "")
    rr.setdefault("Secondary_Label_Tagged_Text", "")

    # -------- Driver fields (new in V4.5) --------
    rr.setdefault("Driver", "")
    rr.setdefault("Driver_Source", "")
    rr.setdefault("Driver_Label_Selected", rr.get("Driver", ""))
    rr.setdefault("Driver_reason", "")

    # -------- Audit convenience --------
    if "Intent_Path" not in rr and "intent_Path_field_aware" in rr:
        rr["Intent_Path"] = clean_cell_no_brackets(rr.get("intent_Path_field_aware", ""))

    return rr

# -----------------------------
# Load inputs
# -----------------------------
snapshots = read_all_jsonl(SNAPSHOT_DIR)
episodes  = read_all_jsonl(CCE_ENRICHED_DIR)
print(f"Loaded {len(snapshots)} snapshots from {SNAPSHOT_DIR}; {len(episodes)} enriched episode rows from {CCE_ENRICHED_DIR}.")

# -----------------------------
# Write CSV (snapshots) — flatten features to JSON string
# -----------------------------
snap_csv = COMBINE_DIR / "snapshots_combined.csv"
if snapshots:
    snaps_flat = []
    for r in snapshots:
        feats = r.get("features", {})
        rr = {**r, "features_json": to_json(feats)}
        rr.pop("features", None)  # drop raw features to avoid CSV/Parquet issues
        snaps_flat.append(rr)

    keys = sorted(set().union(*[set(x.keys()) for x in snaps_flat]))
    with snap_csv.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for r in snaps_flat:
            r_clean = {k: clean_cell_no_brackets(v) for k, v in r.items()}
            w.writerow(r_clean)
    print(f"[ok] {snap_csv}")
else:
    print("[warn] No snapshots found.")

# -----------------------------
# Write CSV (episodes enriched) — align to Step-2 V4.5 schema
# -----------------------------
cce_csv = COMBINE_DIR / "episodes_enriched_combined.csv"
if episodes:
    # Step-2 (V4.5) emitted columns (exact names/casing) + _source_file from our loader.
    SUPPORTED_COLS = [
        # --- Metadata (context)
        "repo", "sha", "prev_sha", "timestamp_epoch_utc", "timestamp_utc", "path", "subject", "_source_file",

        # --- Primary (WHAT)
        "Field", "Primary_Label", "Primary_Label_Category",

        # --- Diff / details
        "old_value", "new_value", "change_type", "change_op", "magnitude",
        "added_items", "removed_items", "repeat_index", "repeat_label",

        # --- Per-source intents (legacy & canonical)
        "intent_Delta", "intent_subject", "intent_Path", "intent_Path_field_aware",
        "intent_delta", "intent_pfa", "intent_path",  # canonical strings (intent_subject already above)

        # --- Aggregated Secondary Label(s)
        "Secondary_Label", "Secondary_Label_Tagged", "Secondary_Label_Tagged_Text",

        # --- Aux & audit
        "Aux_Tags", "Intent_Path", "Intent_Path_Reason", "Intent_Subject_Reason",

        # --- Driver (final 7-bucket intention)
        "Driver", "Driver_Source", "Driver_Label_Selected", "Driver_reason",
    ]

    # Column order requested
    META_COLS = [
        "repo", "_source_file", "sha", "prev_sha", "timestamp_utc", "timestamp_epoch_utc", "path", "subject"
    ]
    ANYTHING_ELSE = [
        "old_value", "new_value", "change_type", "change_op", "magnitude",
        "added_items", "removed_items", "repeat_index", "repeat_label",
        "Intent_Path", "Intent_Path_Reason", "Intent_Subject_Reason",
    ]
    PRIMARY_GROUP = ["Field", "Primary_Label", "Primary_Label_Category"]
    DRIVER_GROUP  = ["Driver", "Driver_Source", "Driver_Label_Selected", "Driver_reason"]
    SECONDARY_GROUP = [
        "Secondary_Label", "Secondary_Label_Tagged", "Secondary_Label_Tagged_Text",
        # canonical per-source strings first (nice for BI tools)
        "intent_delta", "intent_pfa", "intent_path", "intent_subject",
        # legacy per-source for back-compat
        "intent_Delta", "intent_Path", "intent_Path_field_aware",
        "Aux_Tags",
    ]

    ORDERED_COLS = META_COLS + ANYTHING_ELSE + PRIMARY_GROUP + DRIVER_GROUP + SECONDARY_GROUP

    # Build filtered rows (drop anything not in SUPPORTED_COLS); map legacy names → new
    episodes_filtered: List[dict] = []
    for r in episodes:
        r = normalize_episode_row_names(r)

        # filter to supported keys only, keep blank string if truly missing
        rr = {k: r.get(k, "") for k in SUPPORTED_COLS if (k in r) or (k in ORDERED_COLS)}

        # ensure change_op present (older Step-2 versions may lack it)
        if not rr.get("change_op"):
            rr["change_op"] = conservative_change_op(r.get("old_value"), r.get("new_value"))

        # scrub all values to ensure no bracketed list strings survive
        rr = {k: clean_cell_no_brackets(v) for k, v in rr.items()}

        episodes_filtered.append(rr)

    # Final write: only include columns that appear in at least one row
    present_cols = [c for c in ORDERED_COLS if any(c in r and r[c] != "" for r in episodes_filtered)]
    # Always keep the full metadata even if blanks
    for c in META_COLS:
        if c not in present_cols:
            present_cols.insert(0 if c == "repo" else len(present_cols), c)

    with cce_csv.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=present_cols)
        w.writeheader()
        for r in episodes_filtered:
            row = {k: r.get(k, "") for k in present_cols}
            w.writerow(row)
    print(f"[ok] {cce_csv}")
else:
    print("[warn] No enriched episodes found.")

# -----------------------------
# Optional: Parquet with pandas (safe dtypes for slim schema)
# -----------------------------
try:
    import pandas as pd

    # Snapshots → Parquet
    if snap_csv.exists():
        df_s = pd.read_csv(snap_csv, dtype=str, keep_default_na=False)

        # Coerce snapshot epoch timestamps to numeric if present
        for col in ("timestamp", "timestamp_epoch_utc"):
            if col in df_s.columns:
                df_s[col] = pd.to_numeric(df_s[col], errors="coerce").astype("Int64")

        df_s.to_parquet(COMBINE_DIR / "snapshots_combined.parquet", index=False)
        print("[ok] Parquet: snapshots_combined.parquet")

    # Enriched episodes → Parquet
    if cce_csv.exists():
        df_e = pd.read_csv(cce_csv, dtype=str, keep_default_na=False)

        # Stringy columns (safe)
        str_cols = [
            "repo","_source_file","sha","prev_sha","timestamp_utc","path","subject",
            "Field","Primary_Label","Primary_Label_Category",
            "old_value","new_value","change_type","change_op","added_items","removed_items","repeat_label",
            "Secondary_Label","Secondary_Label_Tagged","Secondary_Label_Tagged_Text",
            "intent_delta","intent_pfa","intent_path","intent_subject",
            "intent_Delta","intent_Path","intent_Path_field_aware","Aux_Tags",
            "Intent_Path","Intent_Path_Reason","Intent_Subject_Reason",
            "Driver","Driver_Source","Driver_Label_Selected","Driver_reason",
        ]
        for col in str_cols:
            if col in df_e.columns:
                df_e[col] = df_e[col].astype("string")

        # Numeric-friendly columns
        for num_col in ("timestamp_epoch_utc","magnitude","repeat_index"):
            if num_col in df_e.columns:
                df_e[num_col] = pd.to_numeric(df_e[num_col], errors="coerce")

        # Nullable ints for indices/counters if present
        for i_col in ("timestamp_epoch_utc","repeat_index"):
            if i_col in df_e.columns:
                df_e[i_col] = df_e[i_col].astype("Int64")

        df_e.to_parquet(COMBINE_DIR / "episodes_enriched_combined.parquet", index=False)
        print("[ok] Parquet: episodes_enriched_combined.parquet")

except Exception as e:
    print(f"[note] Skipping Parquet (pandas/pyarrow issue?): {e}")


Loaded 185593 snapshots from C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshots; 13088 enriched episode rows from C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\cce_enriched_V4.4.
[ok] C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\combined_V4.4\snapshots_combined.csv
[ok] C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\combined_V4.4\episodes_enriched_combined.csv
[ok] Parquet: snapshots_combined.parquet
[ok] Parquet: episodes_enriched_combined.parquet
